# Analyze `agent_executor_v1` Output Keys

`llm_api_history.pkl`에서 `Possible_Answer_Docs`의 key가 실제로 어떻게 나오는지 집계합니다.

- 경로만 바꿔서 재사용하세요.
- 핵심 출력: key 빈도, key-set 빈도, non-standard key 샘플.


In [2]:
import json
import pickle
from collections import Counter

import pandas as pd
from pathlib import Path
try:
    from json_repair import repair_json
except Exception:
    repair_json = None

HISTORY_PATH = "../results/BRIGHT/psychology/round5/S=psychology-TV=bottom-up-TPV=5-RInTP=/1-NumLC=10-PlTau=5.0-RCF=0.5-LlmApiB=vllm/Llm=Qwen3-4B-Instruct-2507-NumI=10/NumES=1000-MaxBS=10-S=round5_mrr_selector_accum-FT=1000-GBT=10/PreFRS=branch-RPN=agent_executor_v1-RM=concat-RE=1-RCT=5/RCS=mixed-RGT=10-RRrfK=60-RRC=leaf-REM=replace/llm_api_history.pkl"


# Simple path fallback for different notebook cwd.
history_candidates = [
    Path(HISTORY_PATH),
    Path(HISTORY_PATH.lstrip("../")),
    Path("/data4/jongho/lattice") / HISTORY_PATH.lstrip("../"),
]
resolved = None
for p in history_candidates:
    if p.exists():
        resolved = p
        break
if resolved is None:
    raise FileNotFoundError("History file not found. checked=" + ", ".join(str(x) for x in history_candidates))

rows = pickle.load(open(resolved, "rb"))
print("history_path:", resolved)
print("loaded:", len(rows))


def extract_json_text(text: str) -> str:
    t = str(text or "").strip()
    if "```" in t:
        parts = t.split("```")
        fenced = [parts[i] for i in range(1, len(parts), 2)]
        if fenced:
            t = fenced[-1].strip()
            if t.lower().startswith("json"):
                t = t[4:].strip()
    if "{" in t and "}" in t:
        t = t[t.find("{"): t.rfind("}") + 1]
    return t


def parse_response_to_obj(response_text: str):
    t = extract_json_text(response_text)
    try:
        return json.loads(t)
    except Exception:
        if repair_json is not None:
            try:
                return repair_json(t, return_objects=True)
            except Exception:
                return None
        return None


def find_docs_obj(obj):
    if isinstance(obj, dict):
        if "Possible_Answer_Docs" in obj:
            return obj["Possible_Answer_Docs"]
        if "possible_answer_docs" in obj:
            return obj["possible_answer_docs"]
        for v in obj.values():
            found = find_docs_obj(v)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for it in obj:
            found = find_docs_obj(it)
            if found is not None:
                return found
    return None


key_counter = Counter()
keyset_counter = Counter()
nonstandard_examples = []

std_keys = {"Theory", "Entity", "Example", "Other", "theory", "entity", "example", "other"}

parsed = 0
with_docs = 0

for row in rows:
    obj = parse_response_to_obj(row.get("response", ""))
    if obj is None:
        continue
    parsed += 1

    docs = find_docs_obj(obj)
    if docs is None:
        continue
    with_docs += 1

    if isinstance(docs, dict):
        keys = [str(k).strip() for k in docs.keys() if str(k).strip()]
        for k in keys:
            key_counter[k] += 1
        keyset = tuple(sorted(keys))
        keyset_counter[keyset] += 1

        if any(k not in std_keys for k in keys):
            if len(nonstandard_examples) < 10:
                nonstandard_examples.append(keys)

print("parsed_json:", parsed)
print("with_possible_answer_docs:", with_docs)
print("unique_keys:", len(key_counter))

print("\nTop keys")
print(pd.DataFrame(key_counter.most_common(), columns=["key", "count"]))

print("\nTop key sets")
keyset_rows = [
    {"key_set": " | ".join(kset), "count": cnt}
    for kset, cnt in keyset_counter.most_common(20)
]
print(pd.DataFrame(keyset_rows))

print("\nNon-standard key examples")
print(nonstandard_examples if nonstandard_examples else "None")


loaded: 815
parsed_json: 815
with_possible_answer_docs: 815
unique_keys: 4

Top keys
       key  count
0   Theory    815
1   Entity    815
2  Example    815
3    Other    815

Top key sets
                             key_set  count
0  Entity | Example | Other | Theory    815

Non-standard key examples
None
